# 02: Gold Layer - Cross-sectional Heuristics

**Objective:** Collapse the temporal dimension strategically to create a tabular baseline for each physician (Gold Layer).

**Key Actions:**
- Load Silver Layer.
- Extract cross-sectional metrics: Volatility (Kurtosis/Variance), Recent Trend (EWMA over weeks 75-86), and Fast Seasonality (FFT magnitudes).
- Export the heuristic tabular matrix.

In [ ]:
import pandas as pd
import numpy as np
from scipy.fft import fft
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# silver_df = pd.read_parquet('data/silver/silver_layer.parquet')

## 1. Feature Engineering: Volatility & Trends

In [ ]:
def extract_temporal_heuristics(df: pd.DataFrame, id_col: str = 'NUEVO_ID', list_of_features: list = ['TRX_WEEKLY']) -> pd.DataFrame:
    """
    Computes tabular cross-sectional indicators per physician.
    Assumes data is strictly sorted by ID and Time.
    """
    logger.info("Extracting temporal heuristics (Volatility and Recency)...")
    if df.empty: return pd.DataFrame()
    
    # Baseline groupby
    grouped = df.groupby(id_col)
    
    heuristic_df = pd.DataFrame(index=df[id_col].unique())
    heuristic_df.index.name = id_col
    
    for col in list_of_features:
        if col not in df.columns: continue
        
        # Volatility
        heuristic_df[f'{col}_variance'] = grouped[col].var().fillna(0)
        heuristic_df[f'{col}_kurtosis'] = grouped[col].apply(pd.Series.kurt).fillna(0)
        
        # Recent Trends (Assuming exactly 86 weeks, extracting last 12 weeks: 75-86)
        # We can extract the tail of each group
        recent_data = grouped[col].tail(12).groupby(df[id_col])
        
        # Simple moving average of the tail
        heuristic_df[f'{col}_recent_mean'] = recent_data.mean()
        
        # Exponential Weighted Moving Average for the very last elements
        heuristic_df[f'{col}_ewma_tail'] = grouped[col].apply(
            lambda x: x.tail(12).ewm(span=4, adjust=False).mean().iloc[-1] if len(x) >= 12 else 0
        )
        
    logger.info("Standard heuristics completed.")
    return heuristic_df.reset_index()

# heuristics_p1 = extract_temporal_heuristics(silver_df, list_of_features=['TRX_WEEKLY'])

## 2. Fast Fourier Transform (FFT) for Fast Seasonality

In [ ]:
def extract_fft_features(df: pd.DataFrame, id_col: str = 'NUEVO_ID', col: str = 'TRX_WEEKLY') -> pd.DataFrame:
    """
    Extracts the dominating frequencies' magnitudes to capture cyclic behavior.
    """
    logger.info(f"Extracting FFT coefficients for {col}...")
    if df.empty or col not in df.columns: return pd.DataFrame()
    
    def get_top_fft_magnitudes(series, n_top=3):
        # Compute FFT, ignore DC component (0th index)
        f_vals = np.abs(fft(series.values))[1:len(series)//2]
        # Pad if necessary
        if len(f_vals) < n_top:
            f_vals = np.pad(f_vals, (0, max(0, n_top - len(f_vals))))
        # Sort descending and get top magnitudes
        return np.sort(f_vals)[-n_top:][::-1]
    
    fft_df = df.groupby(id_col)[col].apply(get_top_fft_magnitudes).apply(pd.Series)
    fft_df = fft_df.rename(columns={i: f'{col}_fft_mag_{i+1}' for i in fft_df.columns})
    
    logger.info("FFT feature extraction completed.")
    return fft_df.reset_index()

# fft_features = extract_fft_features(silver_df, col='TRX_WEEKLY')
# gold_features = pd.merge(heuristics_p1, fft_features, on='NUEVO_ID', how='inner')
# gold_features.to_parquet('data/gold/gold_heuristic_features.parquet', index=False)